# Intelligent Tone Pipeline & Detoxifier
This notebook walks through building a dual-model pipeline from scratch:
1. **Tone Classifier:** Identifies whether text is Toxic, Polite, Negative, or Positive.
2. **Seq2Seq Detoxifier:** An Encoder-Decoder model that automatically rewrites toxic text into polite text.
3. **Interactive UI:** Deploys an inline Gradio application to test the pipeline.


### Step 1: Install Dependencies and Import Libraries


In [ ]:
!pip install -q torch datasets gradio

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
import re
import random
import gradio as gr

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

HIDDEN_SIZE = 256
EMBEDDING_DIM = 128
MAX_LENGTH = 50


### Step 2: Define Model Architectures
Here we define the Deep Learning classes previously split across multiple files.


In [ ]:
class ToneClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_classes, dropout_p=0.2):
        super(ToneClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.dropout = nn.Dropout(dropout_p)
        self.gru = nn.GRU(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        _, hidden = self.gru(embedded)
        hidden = hidden.squeeze(0)
        output = self.fc(hidden)
        return output

class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input_tensor):
        embedded = self.dropout(self.embedding(input_tensor))
        output, hidden = self.gru(embedded)
        return output, hidden

class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, max_length=50, dropout_p=0.1):
        super(AttnDecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.max_length = max_length

        self.embedding = nn.Embedding(self.output_size, self.hidden_size)
        self.attn = nn.Linear(self.hidden_size * 2, self.max_length)
        self.attn_combine = nn.Linear(self.hidden_size * 2, self.hidden_size)
        self.dropout = nn.Dropout(dropout_p)
        self.gru = nn.GRU(self.hidden_size, self.hidden_size, batch_first=True)
        self.out = nn.Linear(self.hidden_size, self.output_size)

    def forward(self, input_tensor, hidden, encoder_outputs):
        embedded = self.dropout(self.embedding(input_tensor))
        attn_weights = F.softmax(self.attn(torch.cat((embedded.squeeze(1), hidden.squeeze(0)), 1)), dim=1)

        seq_len = encoder_outputs.shape[1]
        if seq_len < self.max_length:
            padding = torch.zeros(
                encoder_outputs.shape[0], self.max_length - seq_len, self.hidden_size, device=encoder_outputs.device
            )
            encoder_outputs = torch.cat((encoder_outputs, padding), dim=1)
        elif seq_len > self.max_length:
            encoder_outputs = encoder_outputs[:, :self.max_length, :]

        attn_applied = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs)
        output = torch.cat((embedded.squeeze(1), attn_applied.squeeze(1)), 1)
        output = self.attn_combine(output).unsqueeze(1)
        output = F.relu(output)
        output, hidden = self.gru(output, hidden)
        output = F.log_softmax(self.out(output.squeeze(1)), dim=1)

        return output, hidden, attn_weights

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, source, target, start_tokens, teacher_forcing_ratio=0.5):
        batch_size = source.shape[0]
        target_len = target.shape[1]
        target_vocab_size = self.decoder.output_size

        outputs = torch.zeros(batch_size, target_len, target_vocab_size).to(self.device)
        encoder_outputs, hidden = self.encoder(source)
        decoder_input = start_tokens

        for t in range(1, target_len):
            output, hidden, _ = self.decoder(decoder_input, hidden, encoder_outputs)
            outputs[:, t, :] = output

            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            decoder_input = target[:, t].unsqueeze(1) if teacher_force else top1.unsqueeze(1)

        return outputs


### Step 3: Classifier Dataset & Vocabulary Preparation
Preparing the Jigsaw and Yelp datasets for the 4-way Tone Classifier.


In [ ]:
SAMPLES_PER_CLASS = 10000

class ClassifierVocab:
    def __init__(self):
        self.word2index = {"<PAD>": 0, "<UNK>": 1}
        self.n_words = 2

    def normalize_string(self, s):
        s = str(s).lower().strip()
        s = re.sub(r"([.!?])", r" \1", s)
        s = re.sub(r"[^a-zA-Z.!?]+", r" ", s)
        return s

    def add_sentence(self, sentence):
        for word in self.normalize_string(sentence).split(' '):
            if word not in self.word2index:
                self.word2index[word] = self.n_words
                self.n_words += 1

    def sentence_to_tensor(self, sentence, max_len=50):
        sentence = self.normalize_string(sentence)
        indexes = [self.word2index.get(word, self.word2index["<UNK>"]) for word in sentence.split(' ')[:max_len]]
        return torch.tensor(indexes, dtype=torch.long)

def prepare_classifier_data():
    print(f"Downloading datasets and sampling {SAMPLES_PER_CLASS} per class...")
    jigsaw = load_dataset("thesofakillers/jigsaw-toxic-comment-classification-challenge", split="train")
    toxic_texts = jigsaw.filter(lambda x: x['toxic'] == 1)['comment_text'][:SAMPLES_PER_CLASS]
    polite_texts = jigsaw.filter(lambda x: x['toxic'] == 0)['comment_text'][:SAMPLES_PER_CLASS]

    yelp = load_dataset("fancyzhx/yelp_polarity", split="train")
    negative_texts = yelp.filter(lambda x: x['label'] == 0)['text'][:SAMPLES_PER_CLASS]
    positive_texts = yelp.filter(lambda x: x['label'] == 1)['text'][:SAMPLES_PER_CLASS]

    dataset_tuples = []
    dataset_tuples.extend([(text, 0) for text in toxic_texts])
    dataset_tuples.extend([(text, 1) for text in polite_texts])
    dataset_tuples.extend([(text, 2) for text in negative_texts])
    dataset_tuples.extend([(text, 3) for text in positive_texts])
    return dataset_tuples

class TextClassificationDataset(Dataset):
    def __init__(self, data_tuples):
        self.data = data_tuples
        self.vocab = ClassifierVocab()
        print("Building vocabulary...")
        for text, _ in self.data:
            self.vocab.add_sentence(text)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text, label = self.data[idx]
        text_tensor = self.vocab.sentence_to_tensor(text)
        return text_tensor, torch.tensor(label, dtype=torch.long)

def collate_classifier(batch):
    texts, labels = zip(*batch)
    texts_padded = pad_sequence(texts, padding_value=0, batch_first=True)
    labels = torch.stack(labels)
    return texts_padded, labels


### Step 4: Classifier Training Loop


In [ ]:
def train_classifier(batch_size=64, lr=0.001, epochs=10):
    data_tuples = prepare_classifier_data()
    dataset = TextClassificationDataset(data_tuples)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_classifier)

    print(f"Vocab Size: {dataset.vocab.n_words} | Total Samples: {len(dataset)}")

    model = ToneClassifier(vocab_size=dataset.vocab.n_words, embedding_dim=EMBEDDING_DIM, hidden_size=HIDDEN_SIZE, num_classes=4).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(epochs):
        total_loss, correct, total = 0, 0, 0
        for batch_texts, batch_labels in dataloader:
            batch_texts, batch_labels = batch_texts.to(DEVICE), batch_labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(batch_texts)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += batch_labels.size(0)
            correct += (predicted == batch_labels).sum().item()

        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(dataloader):.4f} | Accuracy: {(correct/total)*100:.2f}%")

    print("Saving Classifier model and vocab...")
    torch.save(model.state_dict(), "tone_classifier.pth")
    torch.save(dataset.vocab, "classifier_vocab.pth")


### Step 5: Seq2Seq Translator Dataset Preparation


In [ ]:
class Seq2SeqVocab:
    def __init__(self):
        self.word2index = {"<PAD>": 0, "<EOS>": 1, "<UNK>": 2, "<SOS_POLITE>": 3}
        self.index2word = {0: "<PAD>", 1: "<EOS>", 2: "<UNK>", 3: "<SOS_POLITE>"}
        self.n_words = 4

    def normalize_string(self, s):
        s = str(s).lower().strip()
        s = re.sub(r"([.!?])", r" \1", s)
        s = re.sub(r"[^a-zA-Z.!?]+", r" ", s)
        return s

    def add_sentence(self, sentence):
        for word in self.normalize_string(sentence).split(' '):
            if word not in self.word2index:
                self.word2index[word] = self.n_words
                self.index2word[self.n_words] = word
                self.n_words += 1

    def sentence_to_tensor(self, sentence, start_token=None):
        sentence = self.normalize_string(sentence)
        indexes = []
        if start_token:
            indexes.append(self.word2index[start_token])
        for word in sentence.split(' ')[:MAX_LENGTH - 2]:
            indexes.append(self.word2index.get(word, self.word2index["<UNK>"]))
        indexes.append(self.word2index["<EOS>"])
        return torch.tensor(indexes, dtype=torch.long)

class ParadetoxDataset(Dataset):
    def __init__(self):
        print("Downloading Paradetox dataset...")
        ds = load_dataset("s-nlp/paradetox", split="train")
        self.pairs = []
        self.vocab = Seq2SeqVocab()

        print("Building Seq2Seq vocabulary from parallel pairs...")
        for row in ds:
            toxic, polite = row['en_toxic_comment'], row['en_neutral_comment']
            self.vocab.add_sentence(toxic)
            self.vocab.add_sentence(polite)
            self.pairs.append((toxic, polite))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        toxic, polite = self.pairs[idx]
        source_tensor = self.vocab.sentence_to_tensor(toxic)
        target_tensor = self.vocab.sentence_to_tensor(polite, start_token="<SOS_POLITE>")
        return source_tensor, target_tensor

def collate_seq2seq(batch):
    sources, targets = zip(*batch)
    sources_padded = pad_sequence(sources, padding_value=0, batch_first=True)
    targets_padded = pad_sequence(targets, padding_value=0, batch_first=True)
    return sources_padded, targets_padded


### Step 6: Seq2Seq Translator Training Loop


In [ ]:
def train_seq2seq(batch_size=32, lr=0.001, epochs=15):
    dataset = ParadetoxDataset()
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_seq2seq)

    print(f"Vocab Size: {dataset.vocab.n_words} | Parallel Sentences: {len(dataset)}")

    encoder = EncoderRNN(input_size=dataset.vocab.n_words, hidden_size=HIDDEN_SIZE).to(DEVICE)
    decoder = AttnDecoderRNN(hidden_size=HIDDEN_SIZE, output_size=dataset.vocab.n_words, max_length=MAX_LENGTH).to(DEVICE)
    model = Seq2Seq(encoder, decoder, DEVICE).to(DEVICE)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(ignore_index=0)

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch_idx, (source, target) in enumerate(dataloader):
            source, target = source.to(DEVICE), target.to(DEVICE)

            optimizer.zero_grad()
            start_tokens = target[:, 0].unsqueeze(1)
            output = model(source, target, start_tokens)

            output_dim = output.shape[-1]
            output = output[:, 1:].reshape(-1, output_dim)
            target = target[:, 1:].reshape(-1)

            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} | Average Loss: {total_loss/len(dataloader):.4f}")

    print("Saving Seq2Seq models...")
    torch.save(model.state_dict(), "tone_translator.pth")
    torch.save(dataset.vocab, "seq2seq_vocab.pth")


### Step 7: Execution Menu
Run the cell below to trigger the dataset downloads and start training. **Note:** Only run this once to generate the `.pth` files.


In [ ]:
print("--- Initiating Classifier Training ---")
train_classifier(epochs=10)

print("\n--- Initiating Seq2Seq Detoxifier Training ---")
train_seq2seq(epochs=15)


### Step 8: Inline Gradio Inference Pipeline
Loads the `.pth` files generated in Step 7 and boots up an interactive UI directly inside the notebook.


In [ ]:
CLASS_LABELS = {0: "Toxic 🤬", 1: "Polite 🤝", 2: "Negative 📉", 3: "Positive 🌟"}

classifier_model, classifier_vocab = None, None
translator_model, seq2seq_vocab = None, None
models_loaded = False

try:
    classifier_vocab = torch.load("classifier_vocab.pth", map_location=DEVICE, weights_only=False)
    classifier_model = ToneClassifier(vocab_size=classifier_vocab.n_words, embedding_dim=EMBEDDING_DIM, hidden_size=HIDDEN_SIZE, num_classes=4).to(DEVICE)
    classifier_model.load_state_dict(torch.load("tone_classifier.pth", map_location=DEVICE))
    classifier_model.eval()

    seq2seq_vocab = torch.load("seq2seq_vocab.pth", map_location=DEVICE, weights_only=False)
    encoder = EncoderRNN(input_size=seq2seq_vocab.n_words, hidden_size=HIDDEN_SIZE).to(DEVICE)
    decoder = AttnDecoderRNN(hidden_size=HIDDEN_SIZE, output_size=seq2seq_vocab.n_words, max_length=MAX_LENGTH).to(DEVICE)
    translator_model = Seq2Seq(encoder, decoder, DEVICE).to(DEVICE)
    translator_model.load_state_dict(torch.load("tone_translator.pth", map_location=DEVICE))
    translator_model.eval()
    
    print("✅ Full Pipeline Loaded Successfully.")
    models_loaded = True

except FileNotFoundError:
    print("⚠️ Warning: Pre-trained weights (.pth) not found. Please run the training cell (Step 7) first.")

def process_text(user_text):
    if not models_loaded:
         return "Error", "Models not found.", "Please run the training cell first."
    if not user_text.strip():
        return "Please enter text.", "", "N/A"

    with torch.no_grad():
        class_tensor = classifier_vocab.sentence_to_tensor(user_text).unsqueeze(0).to(DEVICE)
        class_logits = classifier_model(class_tensor)
        predicted_id = class_logits.argmax(1).item()
        detected_tone = CLASS_LABELS.get(predicted_id, "Unknown")

        if predicted_id == 0:
            trans_tensor = seq2seq_vocab.sentence_to_tensor(user_text).unsqueeze(0).to(DEVICE)
            encoder_outputs, hidden = translator_model.encoder(trans_tensor)

            sos_token_id = seq2seq_vocab.word2index["<SOS_POLITE>"]
            decoder_input = torch.tensor([[sos_token_id]], device=DEVICE)
            decoded_words = []

            for t in range(MAX_LENGTH):
                output, hidden, _ = translator_model.decoder(decoder_input, hidden, encoder_outputs)
                top1_id = output.argmax(1).item()
                if top1_id == seq2seq_vocab.word2index["<EOS>"]:
                    break
                decoded_words.append(seq2seq_vocab.index2word[top1_id])
                decoder_input = torch.tensor([[top1_id]], device=DEVICE)

            final_translation = " ".join(decoded_words)
            action_taken = "Rewritten for politeness."
        else:
            final_translation = user_text
            action_taken = "No change needed."

    return detected_tone, action_taken, final_translation

interface = gr.Interface(
    fn=process_text,
    inputs=gr.Textbox(lines=3, placeholder="Type a message...", label="Input Text"),
    outputs=[
        gr.Textbox(label="1. Detected Tone"),
        gr.Textbox(label="2. Pipeline Action"),
        gr.Textbox(label="3. Final Output (Detoxified if necessary)")
    ],
    title="Intelligent Tone Pipeline",
    description="Classifies the tone. If text is classified as 'Toxic 🤬', the Seq2Seq deep learning translator automatically rewrites it to be polite.",
    theme="default"
)

interface.launch(inline=True, share=False)
